# **Data Merging and Feature Engineering**

### Importing Libraries

In [1]:
import pandas as pd

### Importing Datasets

In [18]:
superstore_df = pd.read_csv(
    "../data/raw/superstore.csv",
    encoding="latin1"
)

retail_df = pd.read_csv(
    "../data/raw/retail_price_optimization.csv"
)

survey_df = pd.read_csv("../data/raw/retail_price_survey.csv")

### Create Primary Fact Table

In [5]:
merged_df = superstore_df.copy()
display(merged_df)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,9990,CA-2014-110422,1/21/2014,1/23/2014,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,...,33180,South,FUR-FU-10001889,Furniture,Furnishings,Ultra Door Pull Handle,25.2480,3,0.20,4.1028
9990,9991,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,FUR-FU-10000747,Furniture,Furnishings,Tenex B1-RE Series Chair Mats for Low Pile Car...,91.9600,2,0.00,15.6332
9991,9992,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,TEC-PH-10003645,Technology,Phones,Aastra 57i VoIP phone,258.5760,2,0.20,19.3932
9992,9993,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,OFF-PA-10004041,Office Supplies,Paper,"It's Hot Message Books with Stickers, 2 3/4"" x 5""",29.6000,4,0.00,13.3200


### Feature Engineering

In [11]:
merged_df.columns = (
    merged_df.columns
    .str.lower()
    .str.strip()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)

In [12]:
merged_df['revenue'] = merged_df['sales']

merged_df['units_sold'] = merged_df['quantity']

merged_df['price'] = (
    merged_df['sales'] /
    merged_df['quantity'].replace(0, 1)
)

merged_df['competitor_price'] = (
    merged_df['price'] * 1.05
)

merged_df['discount_percentage'] = (
    merged_df['discount'] * 100
)

merged_df['sales_date'] = (
    merged_df['order_date']
)

### Retail Aggregation

In [17]:
retail_df.columns = (
    retail_df.columns
    .str.lower()
    .str.strip()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)

retail_summary = (
    retail_df.groupby('product_category')['value']
    .mean()
    .reset_index()
)

retail_summary.rename(
    columns={
        'product_category': 'category',
        'value': 'market_average_value'
    },
    inplace=True
)

merged_df = pd.merge(
    merged_df,
    retail_summary,
    on='category',
    how='left'
)

display(merged_df.head())

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,discount,profit,revenue,units_sold,price,competitor_price,discount_percentage,sales_date,market_average_value_x,market_average_value_y
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,0.00,41.9136,261.9600,2,130.9800,137.529000,0.0,11/8/2016,NaN,NaN
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,0.00,219.5820,731.9400,3,243.9800,256.179000,0.0,11/8/2016,NaN,NaN
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,0.00,6.8714,14.6200,2,7.3100,7.675500,0.0,6/12/2016,NaN,NaN
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,0.45,-383.0310,957.5775,5,191.5155,201.091275,45.0,10/11/2015,NaN,NaN
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,0.20,2.5164,22.3680,2,11.1840,11.743200,20.0,10/11/2015,NaN,NaN


### Survey Aggregation

In [21]:
survey_df.columns = (
    survey_df.columns
    .str.lower()
    .str.strip()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)

In [22]:
survey_summary = (
    survey_df
    .groupby('sku_category')
    .agg({
        'sales_amount': 'mean',
        'quantity': 'mean'
    })
    .reset_index()
)

In [23]:
survey_summary.rename(
    columns={
        'sku_category': 'category',
        'sales_amount': 'avg_market_sales',
        'quantity': 'avg_market_quantity'
    },
    inplace=True
)

### Handle Null Values

In [25]:
merged_df.fillna(0, inplace=True)

### Merged DF

In [26]:
merged_df = pd.merge(
    merged_df,
    survey_summary,
    on='category',
    how='left'
)

display(merged_df.head())
print(merged_df.info())

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,price,competitor_price,discount_percentage,sales_date,market_average_value_x,market_average_value_y,avg_market_sales_x,avg_market_quantity_x,avg_market_sales_y,avg_market_quantity_y
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,130.9800,137.529000,0.0,11/8/2016,0.0,0.0,0.0,0.0,NaN,NaN
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,243.9800,256.179000,0.0,11/8/2016,0.0,0.0,0.0,0.0,NaN,NaN
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,7.3100,7.675500,0.0,6/12/2016,0.0,0.0,0.0,0.0,NaN,NaN
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,191.5155,201.091275,45.0,10/11/2015,0.0,0.0,0.0,0.0,NaN,NaN
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,11.1840,11.743200,20.0,10/11/2015,0.0,0.0,0.0,0.0,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 33 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   row_id                  9994 non-null   int64  
 1   order_id                9994 non-null   object 
 2   order_date              9994 non-null   object 
 3   ship_date               9994 non-null   object 
 4   ship_mode               9994 non-null   object 
 5   customer_id             9994 non-null   object 
 6   customer_name           9994 non-null   object 
 7   segment                 9994 non-null   object 
 8   country                 9994 non-null   object 
 9   city                    9994 non-null   object 
 10  state                   9994 non-null   object 
 11  postal_code             9994 non-null   int64  
 12  region                  9994 non-null   object 
 13  product_id              9994 non-null   object 
 14  category                9994 non-null   

In [27]:
merged_df.to_csv(
    '../data/merged/merged_dataset.csv',
    index=False
)